# Manual RLlib Checkpoint Evaluation

Use this notebook to restore an RLlib checkpoint and run the repo's current manual evaluation helper.

This is useful when you want to check whether validation metrics stay the same even after restoring a saved checkpoint.

## What this notebook does

1. Loads the original Hydra config from a run directory.
2. Rebuilds the matching RLlib algorithm.
3. Restores the checkpoint.
4. Runs `_evaluate_with_details(...)` from the current codebase.
5. Prints the aggregate validation summary and the per-seed rows.

Optional:
- override eval seeds or eval episode count
- call `algo.evaluate()` as a secondary comparison
- save outputs to JSON and CSV

In [ ]:
from __future__ import annotations

import json
import sys
from copy import deepcopy
from pathlib import Path
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    pd = None

from omegaconf import OmegaConf


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "sumo_rl").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sumo_rl.experiments.rllib_runner import (
    _build_algorithm_config,
    _evaluate_with_details,
    _restore_checkpoint,
)

print(f"Repo root: {ROOT}")
print(f"pandas available: {pd is not None}")

In [ ]:
# Edit these paths before running.
RUN_DIR = r""
CHECKPOINT_PATH = r""

# Optional overrides.
OVERRIDE_EVAL_SEEDS = None
OVERRIDE_EVAL_EPISODES = None
RAY_NUM_GPUS = 0
RUN_RLLIB_EVALUATE = False
PRINT_FULL_SUMMARY = False

# Optional saving.
SAVE_RESULTS = False
RESULTS_DIR = None

In [ ]:
if not RUN_DIR or not CHECKPOINT_PATH:
    raise ValueError("Set RUN_DIR and CHECKPOINT_PATH first.")

run_dir = Path(RUN_DIR).expanduser().resolve()
checkpoint_path = Path(CHECKPOINT_PATH).expanduser().resolve()
config_path = run_dir / ".hydra" / "config.yaml"

if not run_dir.exists():
    raise FileNotFoundError(f"Run directory does not exist: {run_dir}")
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint path does not exist: {checkpoint_path}")
if not config_path.exists():
    raise FileNotFoundError(f"Hydra config not found: {config_path}")

cfg = OmegaConf.load(config_path)
cfg = deepcopy(cfg)

if OVERRIDE_EVAL_SEEDS is not None:
    override_seeds = [int(seed) for seed in OVERRIDE_EVAL_SEEDS]
    cfg.experiment.eval_seeds = override_seeds
    cfg.experiment.eval_episodes = len(override_seeds)
elif OVERRIDE_EVAL_EPISODES is not None:
    cfg.experiment.eval_episodes = int(OVERRIDE_EVAL_EPISODES)

algorithm_kind = str(cfg.algorithm.kind)

print(f"Run dir: {run_dir}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Algorithm kind: {algorithm_kind}")
print(f"Scenario: {cfg.scenario.name}")
print(f"Eval episodes: {cfg.experiment.eval_episodes}")
print(f"Eval seeds: {cfg.experiment.eval_seeds}")

In [ ]:
import ray

algo = None
summary = None
seed_rows = None
action_plot_rows_by_agent = None
action_timeline_by_agent = None
phase_queue_rows_by_agent = None
tripinfo_distributions = None
raw_rllib_evaluation = None

ray.init(ignore_reinit_error=True, include_dashboard=False, log_to_driver=False, num_gpus=RAY_NUM_GPUS)
try:
    algo_config = _build_algorithm_config(cfg, run_dir, algorithm_kind)
    build_algo = getattr(algo_config, "build_algo", None)
    algo = build_algo() if callable(build_algo) else algo_config.build()

    _restore_checkpoint(algo, checkpoint_path)

    if RUN_RLLIB_EVALUATE and hasattr(algo, "evaluate"):
        raw_rllib_evaluation = algo.evaluate()

    (
        summary,
        seed_rows,
        action_plot_rows_by_agent,
        action_timeline_by_agent,
        phase_queue_rows_by_agent,
        tripinfo_distributions,
    ) = _evaluate_with_details(
        cfg,
        run_dir,
        algo,
        algorithm_kind,
        cfg.logging,
        include_validation_metrics=True,
    )
finally:
    if algo is not None and hasattr(algo, "stop"):
        algo.stop()
    ray.shutdown()

print("Validation summary:")
if PRINT_FULL_SUMMARY:
    pprint(summary)
else:
    key_order = [
        "algorithm/kind",
        "final/eval/mean_reward",
        "final/eval/std_reward",
        "validation/reward_mean",
        "validation/resco_delay_mean",
        "validation/resco_wait_mean",
        "validation/resco_trip_time_mean",
        "validation/resco_queue_mean",
        "validation/resco_tripinfo_count",
        "validation/efficiency_total_arrived",
        "validation/efficiency_total_departed",
    ]
    preview = {key: summary.get(key) for key in key_order if key in summary}
    pprint(preview)

print("\nPer-seed rows:")
pprint(seed_rows)

if raw_rllib_evaluation is not None:
    print("\nRaw algo.evaluate() output:")
    pprint(raw_rllib_evaluation)

In [ ]:
if pd is not None and seed_rows:
    seed_rows_df = pd.DataFrame(seed_rows)
    display(seed_rows_df)
else:
    seed_rows_df = None
    print("pandas is unavailable or there are no seed rows to display.")

if pd is not None and summary is not None:
    summary_df = pd.DataFrame([summary])
    display(summary_df)
else:
    summary_df = None

In [ ]:
if not SAVE_RESULTS:
    print("Set SAVE_RESULTS = True if you want JSON/CSV outputs written to disk.")
else:
    results_dir = Path(RESULTS_DIR).expanduser().resolve() if RESULTS_DIR else (run_dir / "manual_checkpoint_eval")
    results_dir.mkdir(parents=True, exist_ok=True)

    (results_dir / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
    (results_dir / "seed_rows.json").write_text(json.dumps(seed_rows, indent=2, sort_keys=True), encoding="utf-8")
    if raw_rllib_evaluation is not None:
        (results_dir / "rllib_evaluate.json").write_text(
            json.dumps(raw_rllib_evaluation, indent=2, sort_keys=True, default=str),
            encoding="utf-8",
        )

    if pd is not None:
        pd.DataFrame([summary]).to_csv(results_dir / "summary.csv", index=False)
        pd.DataFrame(seed_rows).to_csv(results_dir / "seed_rows.csv", index=False)

    print(f"Saved results to: {results_dir}")